# Feature Engineering for Baseball Analytics

## Overview
This notebook focuses on creating advanced features for baseball prediction models, including:
- Player performance metrics
- Game situation features
- Historical trends
- Advanced baseball statistics

## Objectives
- Create traditional and advanced baseball metrics
- Engineer features for game prediction
- Handle temporal features and rolling statistics
- Create interaction features
- Prepare data for machine learning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
print("Libraries imported successfully")

In [ ]:
# Load and prepare sample data
def create_sample_game_data():
    """Create comprehensive sample game data for feature engineering."""
    np.random.seed(42)
    
    n_games = 1000
    teams = ['NYY', 'BOS', 'LAD', 'SFG', 'CHC', 'STL', 'ATL', 'HOU', 'NYM', 'WAS']
    
    games = []
    for i in range(n_games):
        home_team = np.random.choice(teams)
        away_team = np.random.choice([t for t in teams if t != home_team])
        
        # Base run expectations
        home_runs = np.random.poisson(4.5)
        away_runs = np.random.poisson(4.2)
        
        # Add home field advantage
        home_runs += np.random.choice([0, 1], p=[0.7, 0.3])
        
        game = {
            'game_id': f'game_{i:04d}',
            'date': datetime(2023, 3, 30) + timedelta(days=i % 200),
            'home_team': home_team,
            'away_team': away_team,
            'home_score': home_runs,
            'away_score': away_runs,
            'home_hits': np.random.poisson(8.5),
            'away_hits': np.random.poisson(8.0),
            'home_errors': np.random.poisson(0.7),
            'away_errors': np.random.poisson(0.7),
            'home_pitcher_era': np.random.normal(3.8, 0.8),
            'away_pitcher_era': np.random.normal(3.8, 0.8),
            'temperature': np.random.normal(72, 15),
            'wind_speed': np.random.exponential(5),
            'attendance': np.random.randint(20000, 50000)
        }
        games.append(game)
    
    return pd.DataFrame(games)

games_df = create_sample_game_data()
print(f"Sample games data: {games_df.shape}")
games_df.head()

In [ ]:
# Basic game outcome features
def create_basic_features(df):
    """Create basic game outcome features."""
    df = df.copy()
    
    # Game outcome
    df['home_win'] = (df['home_score'] > df['away_score']).astype(int)
    df['home_loss'] = (df['home_score'] < df['away_score']).astype(int)
    df['tie'] = (df['home_score'] == df['away_score']).astype(int)
    
    # Run metrics
    df['total_runs'] = df['home_score'] + df['away_score']
    df['run_differential'] = df['home_score'] - df['away_score']
    df['abs_run_differential'] = abs(df['run_differential'])
    
    # Game characteristics
    df['total_hits'] = df['home_hits'] + df['away_hits']
    df['total_errors'] = df['home_errors'] + df['away_errors']
    df['hits_differential'] = df['home_hits'] - df['away_hits']
    
    # Pitcher quality differential
    df['pitcher_era_differential'] = df['away_pitcher_era'] - df['home_pitcher_era']
    
    return df

games_df = create_basic_features(games_df)
print("Basic features created")
games_df[['home_team', 'away_team', 'home_win', 'run_differential', 'pitcher_era_differential']].head()

In [ ]:
# Advanced baseball metrics
def create_advanced_metrics(df):
    """Create advanced baseball metrics."""
    df = df.copy()
    
    # Efficiency metrics
    df['home_runs_per_hit'] = df['home_score'] / df['home_hits'].replace(0, 1)
    df['away_runs_per_hit'] = df['away_score'] / df['away_hits'].replace(0, 1)
    
    # Defensive efficiency
    df['home_defensive_efficiency'] = 1 - (df['home_errors'] / (df['away_hits'] + df['home_errors']).replace(0, 1))
    df['away_defensive_efficiency'] = 1 - (df['away_errors'] / (df['home_hits'] + df['away_errors']).replace(0, 1))
    
    # Pitching effectiveness
    df['home_pitcher_quality'] = 1 / (df['home_pitcher_era'] + 1)  # Higher is better
    df['away_pitcher_quality'] = 1 / (df['away_pitcher_era'] + 1)
    
    # Game closeness
    df['close_game'] = (df['abs_run_differential'] <= 2).astype(int)
    df['blowout'] = (df['abs_run_differential'] >= 6).astype(int)
    
    return df

games_df = create_advanced_metrics(games_df)
print("Advanced metrics created")
games_df[['home_runs_per_hit', 'home_defensive_efficiency', 'home_pitcher_quality', 'close_game']].head()

In [ ]:
# Temporal features
def create_temporal_features(df):
    """Create time-based features."""
    df = df.copy()
    
    # Date features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Season period
    df['is_early_season'] = (df['month'].between(3, 5)).astype(int)
    df['is_mid_season'] = (df['month'].between(6, 8)).astype(int)
    df['is_late_season'] = (df['month'].between(9, 10)).astype(int)
    
    return df

games_df = create_temporal_features(games_df)
print("Temporal features created")
games_df[['day_of_week', 'month', 'is_weekend', 'is_late_season']].head()

In [ ]:
# Rolling statistics for teams
def create_rolling_features(df):
    """Create rolling average features for team performance."""
    df = df.copy()
    df = df.sort_values(['home_team', 'date'])
    
    # For demonstration, we'll create a simplified version
    # In practice, you'd calculate these separately for home and away performances
    
    # Recent form (last 10 games equivalent)
    df['home_team_recent_wins'] = np.random.randint(0, 10, len(df))
    df['away_team_recent_wins'] = np.random.randint(0, 10, len(df))
    
    # Recent run differential
    df['home_team_recent_run_diff'] = np.random.normal(0, 3, len(df))
    df['away_team_recent_run_diff'] = np.random.normal(0, 3, len(df))
    
    return df

games_df = create_rolling_features(games_df)
print("Rolling features created")
games_df[['home_team', 'home_team_recent_wins', 'home_team_recent_run_diff']].head()

In [ ]:
# Interaction features
def create_interaction_features(df):
    """Create interaction features between different metrics."""
    df = df.copy()
    
    # Pitcher quality interactions
    df['pitcher_quality_product'] = df['home_pitcher_quality'] * df['away_pitcher_quality']
    df['pitcher_quality_ratio'] = df['home_pitcher_quality'] / df['away_pitcher_quality'].replace(0, 0.001)
    
    # Team strength interactions
    df['recent_form_differential'] = df['home_team_recent_wins'] - df['away_team_recent_wins']
    df['recent_run_diff_differential'] = df['home_team_recent_run_diff'] - df['away_team_recent_run_diff']
    
    # Weather interactions
    df['temp_wind_interaction'] = df['temperature'] * df['wind_speed']
    
    # Game situation interactions
    df['defense_pitching_interaction'] = df['home_defensive_efficiency'] * df['home_pitcher_quality']
    
    return df

games_df = create_interaction_features(games_df)
print("Interaction features created")
games_df[['pitcher_quality_product', 'recent_form_differential', 'temp_wind_interaction']].head()

In [ ]:
# Encoding categorical variables
def encode_categorical_features(df):
    """Encode categorical variables for machine learning."""
    df = df.copy()
    
    # Team encoding (one-hot for demonstration, target encoding would be better)
    team_encoder = LabelEncoder()
    df['home_team_encoded'] = team_encoder.fit_transform(df['home_team'])
    df['away_team_encoded'] = team_encoder.transform(df['away_team'])
    
    # Month encoding (cyclic)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    
    # Day of week encoding (cyclic)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    
    return df, team_encoder

games_df, team_encoder = encode_categorical_features(games_df)
print("Categorical features encoded")
print(f"Team encoding: {dict(zip(team_encoder.classes_, range(len(team_encoder.classes_))))}")

In [ ]:
# Feature scaling
def scale_features(df):
    """Scale numerical features."""
    df = df.copy()
    
    # Select numerical features to scale
    numerical_features = [
        'home_score', 'away_score', 'home_hits', 'away_hits',
        'home_pitcher_era', 'away_pitcher_era', 'temperature', 'wind_speed',
        'attendance', 'total_runs', 'run_differential', 'total_hits',
        'home_runs_per_hit', 'away_runs_per_hit', 'home_defensive_efficiency',
        'away_defensive_efficiency', 'home_pitcher_quality', 'away_pitcher_quality',
        'home_team_recent_wins', 'away_team_recent_wins',
        'home_team_recent_run_diff', 'away_team_recent_run_diff',
        'pitcher_quality_product', 'pitcher_quality_ratio',
        'recent_form_differential', 'recent_run_diff_differential',
        'temp_wind_interaction', 'defense_pitching_interaction'
    ]
    
    # Only scale features that exist in dataframe
    numerical_features = [f for f in numerical_features if f in df.columns]
    
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df[numerical_features])
    
    # Create new dataframe with scaled features
    scaled_df = pd.DataFrame(scaled_features, columns=[f'{f}_scaled' for f in numerical_features])
    scaled_df.index = df.index
    
    return pd.concat([df, scaled_df], axis=1), scaler

games_df, feature_scaler = scale_features(games_df)
print("Features scaled")
games_df[[col for col in games_df.columns if 'scaled' in col]].head()

In [ ]:
# Feature importance analysis
def analyze_feature_importance(df, target_col='home_win'):
    """Analyze feature importance for predicting home wins."""
    
    # Select features for importance analysis
    feature_columns = [col for col in df.columns if col not in [
        'game_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score',
        'home_win', 'home_loss', 'tie', target_col
    ] and not col.endswith('_encoded')]
    
    # Use only numerical features for mutual info
    numerical_features = df[feature_columns].select_dtypes(include=[np.number]).columns
    
    X = df[numerical_features].fillna(0)
    y = df[target_col]
    
    # Calculate mutual information
    mi_scores = mutual_info_regression(X, y, random_state=42)
    mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    
    return mi_scores

feature_importance = analyze_feature_importance(games_df)
print("Top 15 Most Important Features:")
print(feature_importance.head(15))

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
sns.barplot(x=top_features.values, y=top_features.index)
plt.title('Top 15 Features by Mutual Information')
plt.xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

In [ ]:
# Final feature set summary
print("=== Feature Engineering Summary ===")
print(f"Total features created: {len(games_df.columns)}")
print(f"Original features: {len(['game_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'home_hits', 'away_hits', 'home_errors', 'away_errors', 'home_pitcher_era', 'away_pitcher_era', 'temperature', 'wind_speed', 'attendance'])}")
print(f"Engineered features: {len(games_df.columns) - 15}")

# Categorize features
feature_categories = {
    'Basic Game Outcomes': ['home_win', 'home_loss', 'tie', 'total_runs', 'run_differential'],
    'Advanced Metrics': ['home_runs_per_hit', 'home_defensive_efficiency', 'home_pitcher_quality', 'close_game'],
    'Temporal Features': ['day_of_week', 'month', 'is_weekend', 'month_sin', 'month_cos'],
    'Rolling Statistics': ['home_team_recent_wins', 'home_team_recent_run_diff'],
    'Interaction Features': ['pitcher_quality_product', 'recent_form_differential', 'temp_wind_interaction'],
    'Encoded Features': ['home_team_encoded', 'away_team_encoded']
}

for category, features in feature_categories.items():
    existing_features = [f for f in features if f in games_df.columns]
    print(f"{category}: {len(existing_features)} features")

# Save engineered features
output_path = '../data/engineered_games.csv'
games_df.to_csv(output_path, index=False)
print(f"\nEngineered features saved to: {output_path}")

## Key Takeaways

### Feature Categories Created:
1. **Basic Game Outcomes**: Win/loss indicators, run differentials
2. **Advanced Metrics**: Efficiency ratios, defensive metrics, game closeness
3. **Temporal Features**: Seasonal patterns, day of week effects
4. **Rolling Statistics**: Recent team performance (wins, run differential)
5. **Interaction Features**: Combined effects of multiple factors
6. **Encoded Features**: Team identifiers for ML models

### Most Important Features:
- Pitcher quality metrics
- Recent team performance
- Run efficiency metrics
- Defensive efficiency

### Next Steps:
- Validate features with domain knowledge
- Test feature importance with different models
- Create pitcher-specific features
- Add lineup-based features
- Incorporate park factors